# Training data generation for Kissat - Mult CNF using Grid search

In [1]:
from setup import setup_tools, cnf_path
from tools_fns import kissat_check

setup_tools()

Setting project root: /home/krishnendu/Research/fv-invariant-mining
Updated PATH to include: [PosixPath('/home/krishnendu/Research/fv-invariant-mining/.tools/cadical/build'), PosixPath('/home/krishnendu/Research/fv-invariant-mining/.tools/abc'), PosixPath('/home/krishnendu/Research/fv-invariant-mining/.tools/oss-cad-suite/bin'), PosixPath('/home/krishnendu/Research/fv-invariant-mining/.tools/aiger'), PosixPath('/home/krishnendu/Research/fv-invariant-mining/.tools/cadiback'), PosixPath('/home/krishnendu/Research/fv-invariant-mining/.tools/kissat/build')]


In [3]:
kissat_check(cnf_path/"miter_mult_8bit.cnf", timeout=1)

(1.0, 'INDETERMINATE', <utils.TermOutput at 0x7fdc86359d10>)

In [4]:
import os
os.makedirs('/home/krishnendu/Research/fv-invariant-mining/logs/')

FileExistsError: [Errno 17] File exists: '/home/krishnendu/Research/fv-invariant-mining/logs/'

In [5]:
dimacs: Path | str=cnf_path/"miter_mult_5bit.cnf"
rt, _, out = kissat_check(dimacs, {})
print(rt, out.output)

0.03 c ---- [ banner ] ------------------------------------------------------------
c
c Kissat SAT Solver
c 
c Copyright (c) 2021-2024 Armin Biere University of Freiburg
c Copyright (c) 2019-2021 Armin Biere Johannes Kepler University Linz
c 
c Version 4.0.4 8af8e56f174b778aef3aa45af9f739b2a5f492c2
c gcc (GCC) 15.2.1 20260123 (Red Hat 15.2.1-7) -W -Wall -O3 -DNDEBUG
c Thu Apr 23 06:47:51 PM IST 2026 Linux krishnendu-LN 6.19.12-200.fc43.x86_64 x86_64
c
c ---- [ parsing ] -----------------------------------------------------------
c
c opened and reading DIMACS file:
c
c   /home/krishnendu/Research/fv-invariant-mining/data/circuits/cnf/miter_mult_5bit.cnf
c
c parsed 'p cnf 235 876' header
c closing input after reading 12216 bytes (12 KB)
c finished parsing after 0.00 seconds
c
c ---- [ solving ] -----------------------------------------------------------
c
c seconds switched rate     size/glue tier1 binary     remaining
c        MB reductions conflicts size  tier2  irredundant
c         l

In [ ]:
from math import ceil
from pathlib import Path

from ConfigSpace import Configuration, ConfigurationSpace

from smac import HyperparameterOptimizationFacade, Scenario

configspace = ConfigurationSpace(
	{
		'decay': [1] + [i for i in range(10, 201, 10)], 
		'eliminateeffort': [i for i in range(0, 2000, 100)], 
		'stable': [i for i in range(3)]
	})

circuit = cnf_path/"miter_mult_11bit.cnf"
reg_time, _, _ = kissat_check(circuit)
reg_time_fl = reg_time if reg_time else 0.0
timeout=int(ceil(ceil(reg_time_fl)*2.5))

print("Regular time:", reg_time_fl)

max_time_budget = 3600 * 3 # 3 hrs
min_iters = 20
time_budget = max(max_time_budget, min_iters * timeout)
print("Time budget:", time_budget)

def kissat_run(config: Configuration, seed: int, dimacs: Path | str=circuit) -> float:
	rt, status, out = kissat_check(dimacs, config, timeout=timeout) # 2.5 x regular's timeout
	params_list = " ".join([f"--{k}={v}" for k, v in config.items()])
	print(rt, params_list)

	dimacs = Path(dimacs)
	config_str = ",".join(f"{k}-{v}" for k, v in config.items())
	log_fn = f"/home/krishnendu/Research/fv-invariant-mining/logs/kissat-{dimacs.stem}-{config_str}.log"
	with open(log_fn, "w", encoding="utf-8") as file:
		file.write(out.output)
	
	assert rt is not None
	if status == 'INDETERMINATE':
		rt *= 10 # penalize
	return float(rt)

# Scenario object specifying the optimization environment
scenario = Scenario(configspace, deterministic=True, n_trials=40, walltime_limit=time_budget)

# Use SMAC to find the best configuration/hyperparameters
smac = HyperparameterOptimizationFacade(scenario, kissat_run)
incumbent = smac.optimize()

Regular time: 411.64
Time budget: 20600
[INFO][abstract_initial_design.py:91] Reducing the number of initial configurations from 30 to 10 (max_ratio == 0.25).
[WARNING][target_function_runner.py:74] The argument dimacs is not set by SMAC: Consider removing it from the target function.
[INFO][abstract_initial_design.py:143] Using 10 initial design configurations and 0 additional configurations.
[INFO][abstract_intensifier.py:313] Using only one seed for deterministic scenario.


In [16]:
!rm -rf /home/krishnendu/Research/fv-invariant-mining/notebooks/smac3_output

In [12]:
reg_time_fl

2.89